# Parte 2 — Tratamento de dados com pandas

**Objetivo desta parte:** transformar o JSON bruto salvo em `data/raw` em um
`DataFrame` limpo e consistente, praticando os problemas mais comuns de dados do
mundo real: tipos, fusos horários, duplicidade, valores faltantes e outliers.

Partimos exatamente de onde a Parte 1 parou: os arquivos `clima_raw_*.json` em
`data/raw/`.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../../data/raw")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Lendo o JSON bruto como DataFrame

Cada arquivo tem a estrutura de "arrays paralelos" que exploramos na Parte 1
(`hourly.time`, `hourly.temperature_2m`, ...). Para virar um `DataFrame`, cada array
vira uma coluna — é literalmente isso que `pd.DataFrame(dict_de_listas)` faz.


In [2]:
def carregar_cidade(caminho_json: Path) -> pd.DataFrame:

    # Lê um JSON bruto de uma cidade e devolve um DataFrame com uma linha por hora
    with open(caminho_json, encoding="utf-8") as f:
        payload = json.load(f)

    df = pd.DataFrame(payload["hourly"])
    # O nome do arquivo carrega o slug da cidade: clima_raw_<slug>.json
    slug = caminho_json.stem.replace("clima_raw_", "")
    df["cidade"] = slug
    return df

In [3]:
sorted(RAW_DIR.glob("clima_raw_*.json"))

[PosixPath('../../data/raw/clima_raw_belem.json'),
 PosixPath('../../data/raw/clima_raw_manaus.json'),
 PosixPath('../../data/raw/clima_raw_porto_alegre.json'),
 PosixPath('../../data/raw/clima_raw_recife.json'),
 PosixPath('../../data/raw/clima_raw_rio_de_janeiro.json'),
 PosixPath('../../data/raw/clima_raw_sao_paulo.json')]

In [4]:
arquivos_cidades = sorted(RAW_DIR.glob("clima_raw_*.json"))
print("Arquivos encontrados:", [a.name for a in arquivos_cidades])

df = pd.concat(
    [carregar_cidade(a) for a in arquivos_cidades], ignore_index=True
    )

df.shape


Arquivos encontrados: ['clima_raw_belem.json', 'clima_raw_manaus.json', 'clima_raw_porto_alegre.json', 'clima_raw_recife.json', 'clima_raw_rio_de_janeiro.json', 'clima_raw_sao_paulo.json']


(4464, 6)

## 2. Exploração inicial

Antes de tratar qualquer coisa, é preciso conhecer os dados: quantas linhas, quais
tipos, faixas de valores, quantas cidades.


In [5]:
df.head()


,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,cidade
0,2025-01-01T00:00,24.0,97,0.0,2.9,belem
1,2025-01-01T01:00,23.9,98,0.0,1.6,belem
2,2025-01-01T02:00,23.9,99,0.0,3.3,belem
3,2025-01-01T03:00,23.8,98,0.0,3.5,belem
4,2025-01-01T04:00,23.9,97,0.1,4.4,belem


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  4464 non-null   str    
 1   temperature_2m        4464 non-null   float64
 2   relative_humidity_2m  4464 non-null   int64  
 3   precipitation         4464 non-null   float64
 4   wind_speed_10m        4464 non-null   float64
 5   cidade                4464 non-null   str    
dtypes: float64(3), int64(1), str(2)
memory usage: 317.5 KB


In [7]:
df.describe()

,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m
count,4464.000000,4464.000000,4464.000000,4464.000000
mean,25.766555,78.608199,0.228002,7.678203
std,3.229874,14.191496,0.840286,4.203422
min,15.400000,28.000000,0.000000,0.000000
25%,24.000000,70.000000,0.000000,4.500000
50%,25.750000,81.000000,0.000000,7.100000
75%,27.800000,90.000000,0.100000,10.400000
max,38.200000,100.000000,12.000000,23.100000


In [8]:
df["cidade"].value_counts()

cidade
belem             744
manaus            744
porto_alegre      744
recife            744
rio_de_janeiro    744
sao_paulo         744
Name: count, dtype: int64

## 3. Convertendo `time` para `datetime`

A coluna `time` chega como string (`"2025-01-01T00:00"`). Convertemos para
`datetime64` para poder usar toda a API temporal do pandas (`.dt`, `resample`,
`Grouper`, etc.) nas próximas partes.

**Sobre timezone:** pedimos os dados já em `America/Sao_Paulo` na requisição da
Parte 1, então os horários já estão no fuso local de cada cidade *nominalmente*, mas
sem informação de timezone anexada (`tz-naive`). Como estamos comparando cidades
brasileiras que hoje estão todas no mesmo fuso (UTC-3, sem horário de verão desde
2019), deixamos os timestamps **tz-naive** — é uma decisão explícita, não um
esquecimento. Se comparássemos cidades de fusos diferentes, precisaríamos localizar
(`tz_localize`) cada uma antes de comparar.


In [9]:
df["datetime"] = pd.to_datetime(df["time"])
df = df.drop(columns=["time"])
df[["cidade", "datetime"]].head()

,cidade,datetime
0,belem,2025-01-01 00:00:00
1,belem,2025-01-01 01:00:00
2,belem,2025-01-01 02:00:00
3,belem,2025-01-01 03:00:00
4,belem,2025-01-01 04:00:00


In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   temperature_2m        4464 non-null   float64       
 1   relative_humidity_2m  4464 non-null   int64         
 2   precipitation         4464 non-null   float64       
 3   wind_speed_10m        4464 non-null   float64       
 4   cidade                4464 non-null   str           
 5   datetime              4464 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(1), str(1)
memory usage: 247.7 KB


## 4. Checando duplicidade de índice

Cada par (cidade, datetime) deveria ser único — uma leitura por cidade por hora.
Vale sempre checar isso explicitamente antes de seguir, principalmente se o pipeline
puder rodar mais de uma vez sobre o mesmo período (reprocessamento).


In [11]:
print(df["cidade"].unique())
print(df["cidade"].nunique())

<ArrowStringArray>
['belem', 'manaus', 'porto_alegre', 'recife', 'rio_de_janeiro', 'sao_paulo']
Length: 6, dtype: str
6


In [12]:
duplicados = df.duplicated(subset=["cidade", "datetime"]).sum()
print(f"Linhas duplicadas (cidade, datetime): {duplicados}")

Linhas duplicadas (cidade, datetime): 0


## 5. Simulando falhas de sensor (para fins didáticos)

Os dados da Open-Meteo (reanálise ERA5) são, na prática, completos — não há buracos
nem leituras absurdas. Como esta parte precisa treinar tratamento de nulos e outliers,
vamos **injetar de propósito** algumas falhas simuladas, documentadas e com seed
fixa para reprodutibilidade. Em um pipeline real, essa etapa não existiria: os
problemas apareceriam naturalmente na fonte.


In [13]:
rng = np.random.default_rng(seed=42) # cria um gerador de números aleatórios do NumPy

# df_sujo = df não criaria uma cópia — os dois nomes ficariam apontando para o 
# mesmíssimo objeto DataFrame na memória. Qualquer alteração feita em df_tratado
df_sujo = df.copy()

# 1) ~1% das leituras de temperatura e umidade "somem" (sensor sem leitura)
indices_temp = rng.choice(df_sujo.index, size=int(len(df_sujo) * 0.01), replace=False)
df_sujo.loc[indices_temp, "temperature_2m"] = np.nan

indices_umidade = rng.choice(df_sujo.index, size=int(len(df_sujo) * 0.01), replace=False)
df_sujo.loc[indices_umidade, "relative_humidity_2m"] = np.nan

# 2) precipitação: em vez de NaN, o sensor às vezes manda None -> after JSON round-trip
#    isso normalmente aparece como object/NaN misturado com float. Simulamos isso.
indices_precip = rng.choice(df_sujo.index, size=int(len(df_sujo) * 0.02), replace=False)
df_sujo.loc[indices_precip, "precipitation"] = None

# 3) outlier grosseiro de vento (sensor com defeito, ex.: pico de 500 km/h)
indices_vento_outlier = rng.choice(df_sujo.index, size=5, replace=False)
df_sujo.loc[indices_vento_outlier, "wind_speed_10m"] = 500.0

df_sujo.isna().sum()


temperature_2m          44
relative_humidity_2m    44
precipitation           89
wind_speed_10m           0
cidade                   0
datetime                 0
dtype: int64

## 6. Valores faltantes
(Trabalhar este conceito com os alunos)

Estratégia por variável, e por quê:

- **Temperatura** (`temperature_2m`): grandeza física que varia suavemente hora a
  hora → **interpolação temporal** (`interpolate`) é mais fiel do que um simples
  `ffill`.
- **Umidade** (`relative_humidity_2m`): mesmo raciocínio da temperatura.
- **Precipitação** (`precipitation`): a ausência de leitura é tratada aqui como "sem
  chuva registrada" e preenchida com `0.0` — uma decisão *conservadora e discutível*:
  se o sensor realmente falhou durante um evento de chuva, subestimamos o total. Em
  um cenário real, o ideal seria cruzar com uma estação vizinha antes de assumir zero.

Tudo é feito **por cidade** (`groupby("cidade")`), para uma falha em São Paulo não
"vazar" e influenciar a imputação em Manaus.


| `def` | `lambda` |
|---|---|
| `def nome(param):` | `lambda param:` |
| `return expressao` | `expressao` (sem `return`, é implícito) |
| tem nome, pode ter várias linhas | não tem nome, só **uma expressão** |

In [14]:
# Garantir a ordenação primeiro
# interpolate → resolve os "buracos no meio" da série, com uma estimativa mais realista (linear) do que um simples repete-valor.
# ffill → resolve o que sobrou nas pontas finais (não tinha valor futuro para interpolar).
# bfill → resolve o que sobrou nas pontas iniciais (não tinha valor passado para interpolar).

df_sujo = df_sujo.sort_values(["cidade", "datetime"]).reset_index(drop=True) 

df_tratado = df_sujo.copy()

for coluna in ["temperature_2m", "relative_humidity_2m"]:
    df_tratado[coluna] = df_tratado.groupby("cidade")[coluna].transform(
        lambda s: s.interpolate(method="linear").ffill().bfill()
    )

''''
def interpolar_serie(s):
    return s.interpolate(method="linear").ffill().bfill()
'''

df_tratado["precipitation"] = df_tratado["precipitation"].fillna(0.0)

df_tratado.isna().sum()


temperature_2m          0
relative_humidity_2m    0
precipitation           0
wind_speed_10m          0
cidade                  0
datetime                0
dtype: int64

| Onde `transform` existe | O que agrupa | Resultado |
|---|---|---|
| `Series.transform(func)` | nada (a série toda é "um grupo só") | mesmo tamanho da série |
| `DataFrame.transform(func)` | nada (aplica em cada coluna, independente) | mesmo formato do DataFrame |
| `SeriesGroupBy.transform(func)` | por grupo (`groupby(...)[coluna]`) | mesmo tamanho da série original |
| `DataFrameGroupBy.transform(func)` | por grupo (`groupby(...)`, sem selecionar coluna) | mesmo formato do DataFrame original |

## 7. Tipos inconsistentes

Depois de preencher os `None` de precipitação, confirmamos que a coluna é
numérica (`float64`) — o `None` isolado no meio de floats costuma "contaminar" a
coluna com `dtype=object`, o que quebra operações matemáticas silenciosamente até
alguém tentar somar e receber um erro.


In [15]:
print("dtype antes do fillna (df_sujo):", df_sujo["precipitation"].dtype)
print("dtype depois do fillna (df_tratado):", df_tratado["precipitation"].dtype)

df_tratado["precipitation"] = pd.to_numeric(df_tratado["precipitation"], errors="raise")


dtype antes do fillna (df_sujo): float64
dtype depois do fillna (df_tratado): float64


## 8. Outliers

Usamos o método do **IQR (intervalo interquartil)**: qualquer valor fora de
`[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` é considerado suspeito. Aplicamos isso à velocidade
do vento, onde plantamos o outlier de 500 km/h — mas a função é genérica e serve
para qualquer coluna numérica.


In [16]:
def detectar_outliers_iqr(serie: pd.Series) -> pd.Series: #explicar o que é pandas series
    # Retorna uma máscara booleana marcando outliers pelo critério do IQR
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return (serie < limite_inferior) | (serie > limite_superior)


In [17]:
mascara_outliers = df_tratado.groupby("cidade")["wind_speed_10m"].transform(
    lambda s: detectar_outliers_iqr(s)
) 
print(f"Outliers de vento encontrados: {mascara_outliers.sum()}")

Outliers de vento encontrados: 30


In [18]:
df_tratado.loc[mascara_outliers, ["cidade", "datetime", "wind_speed_10m"]]

,cidade,datetime,wind_speed_10m
61,belem,2025-01-03 13:00:00,14.2
84,belem,2025-01-04 12:00:00,13.3
109,belem,2025-01-05 13:00:00,14.0
136,belem,2025-01-06 16:00:00,16.6
200,belem,2025-01-09 08:00:00,500.0
300,belem,2025-01-13 12:00:00,13.0
302,belem,2025-01-13 14:00:00,16.2
303,belem,2025-01-13 15:00:00,16.1
304,belem,2025-01-13 16:00:00,18.0
305,belem,2025-01-13 17:00:00,14.4


In [19]:
# Tratamos o outlier como um dado faltante e reaplicamos a mesma interpolação
# temporal usada para temperatura/umidade — mais coerente do que simplesmente
# descartar a linha inteira (perderíamos as outras variáveis daquela hora).
df_tratado.loc[mascara_outliers, "wind_speed_10m"] = np.nan
df_tratado["wind_speed_10m"] = df_tratado.groupby("cidade")["wind_speed_10m"].transform(
    lambda s: s.interpolate(method="linear").ffill().bfill()
)

df_tratado.loc[mascara_outliers.index[mascara_outliers], ["cidade", "datetime", "wind_speed_10m"]]


,cidade,datetime,wind_speed_10m
61,belem,2025-01-03 13:00:00,8.700000
84,belem,2025-01-04 12:00:00,10.100000
109,belem,2025-01-05 13:00:00,7.150000
136,belem,2025-01-06 16:00:00,3.950000
200,belem,2025-01-09 08:00:00,3.500000
300,belem,2025-01-13 12:00:00,12.050000
302,belem,2025-01-13 14:00:00,12.700000
303,belem,2025-01-13 15:00:00,12.600000
304,belem,2025-01-13 16:00:00,12.500000
305,belem,2025-01-13 17:00:00,12.400000


## 9. Padronizando nomes de colunas

Nomes em snake_case e com a unidade explícita no nome — evita ter que abrir a
documentação da API toda vez que alguém esquecer se `temperature_2m` está em °C ou
°F.


In [20]:
df_tratado = df_tratado.rename(columns={
    "temperature_2m": "temp_c",
    "relative_humidity_2m": "umidade_pct",
    "precipitation": "precipitacao_mm",
    "wind_speed_10m": "vento_kmh",
})

df_tratado = df_tratado[["cidade", "datetime", "temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]]
df_tratado.head()


,cidade,datetime,temp_c,umidade_pct,precipitacao_mm,vento_kmh
0,belem,2025-01-01 00:00:00,24.0,97.0,0.0,2.9
1,belem,2025-01-01 01:00:00,23.9,98.0,0.0,1.6
2,belem,2025-01-01 02:00:00,23.9,99.0,0.0,3.3
3,belem,2025-01-01 03:00:00,23.8,98.0,0.0,3.5
4,belem,2025-01-01 04:00:00,23.9,97.0,0.1,4.4


## 10. Otimizando o uso de memória

Antes de salvar, vale medir quanto de memória o `DataFrame` realmente ocupa — e
por quê. `df.memory_usage()` **sem** `deep=True` subestima colunas de texto: para
colunas `object` (como `cidade`), o pandas guarda só o tamanho dos ponteiros para
as strings, não o das strings em si.

In [21]:
print("memory_usage padrão (deep=False):", f"{df_tratado.memory_usage(deep=False).sum() / 1024:.1f} KB")
print("memory_usage real     (deep=True): ", f"{df_tratado.memory_usage(deep=True).sum() / 1024:.1f} KB")

df_tratado.memory_usage(deep=True)

memory_usage padrão (deep=False): 247.7 KB
memory_usage real     (deep=True):  247.7 KB


Index                132
cidade             74958
datetime           35712
temp_c             35712
umidade_pct        35712
precipitacao_mm    35712
vento_kmh          35712
dtype: int64

A coluna `cidade` sozinha domina o total — apesar de ter só **5 valores únicos**
repetidos milhares de vezes, o pandas guarda cada ocorrência como uma string
Python separada. É exatamente o cenário ideal para o dtype `category`: baixa
cardinalidade (poucos valores distintos frente ao total de linhas).

Regra prática: `category` compensa quando `nunique() / len(df)` é baixo (aqui é
5/3720 ≈ 0.1%); para uma coluna de texto quase toda única (ex.: um ID), `category`
pode até **piorar** a memória, porque soma o custo das categorias com o de um
índice extra por trás dos panos.

Também baixamos a precisão das colunas numéricas de `float64` para `float32` com
`pd.to_numeric(..., downcast="float")` — a Open-Meteo não entrega precisão além
da segunda casa decimal, então `float32` (~7 dígitos significativos) sobra.

In [22]:
df_otimizado = df_tratado.copy()
df_otimizado["cidade"] = df_otimizado["cidade"].astype("category")

colunas_float = ["temp_c", "umidade_pct", "precipitacao_mm", "vento_kmh"]
for coluna in colunas_float:
    df_otimizado[coluna] = pd.to_numeric(df_otimizado[coluna], downcast="float")

df_otimizado.dtypes

cidade                   category
datetime           datetime64[us]
temp_c                    float32
umidade_pct               float32
precipitacao_mm           float32
vento_kmh                 float32
dtype: object

In [23]:
antes = df_tratado.memory_usage(deep=True).sum()
depois = df_otimizado.memory_usage(deep=True).sum()

print(f"Antes:   {antes / 1024:.1f} KB")
print(f"Depois:  {depois / 1024:.1f} KB")
print(f"Redução: {1 - depois / antes:.1%}")

Antes:   247.7 KB
Depois:  109.2 KB
Redução: 55.9%


Numa tabela de 3720 linhas a economia é só um detalhe — mas o ganho **escala com o
tamanho do dado**: o mesmo pipeline rodando para as 27 UFs do Brasil e 5 anos de
histórico (em vez de 5 cidades e 1 mês) teria ~1000x mais linhas, e a mesma técnica
evitaria estourar a memória disponível bem antes de qualquer outra otimização.

Não seguimos com `df_otimizado` para a etapa de salvar: CSV não guarda dtype
(`category`/`float32` viram `object`/`float64` de novo ao reabrir o arquivo com
`pd.read_csv`, como a seção de conferência final abaixo confirma), então o ganho só
vale enquanto o `DataFrame` continua em memória ou é salvo em um formato binário
como [Parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html)
(que preserva os tipos exatamente). Vale a técnica ficar no repertório: é comum
reaplicá-la ao ler o CSV de volta, especificando os dtypes diretamente em
`pd.read_csv(..., dtype={"cidade": "category"})` em vez de otimizar depois de
carregar.

## 11. Salvando o dataset tratado

Esse arquivo é o ponto de partida das próximas partes (agregações, visualização e
persistência em SQLite).


In [24]:
caminho_saida = PROCESSED_DIR / "clima_tratado.csv"
df_tratado.to_csv(caminho_saida, index=False)
print(f"Salvo: {caminho_saida} ({len(df_tratado)} linhas)")


Salvo: ../../data/processed/clima_tratado.csv (4464 linhas)


## 12. Conferência final

Recarregamos o CSV do zero para garantir que ele é auto-suficiente e confere com o
que esperamos — inclusive checando que `datetime` volta como texto (CSV não guarda
tipos) e precisa ser reconvertido por quem for consumir esse arquivo.


In [25]:
conferencia = pd.read_csv(caminho_saida)
print(conferencia.dtypes)
print("\nShape:", conferencia.shape)
conferencia.head()


cidade                 str
datetime               str
temp_c             float64
umidade_pct        float64
precipitacao_mm    float64
vento_kmh          float64
dtype: object

Shape: (4464, 6)


,cidade,datetime,temp_c,umidade_pct,precipitacao_mm,vento_kmh
0,belem,2025-01-01 00:00:00,24.0,97.0,0.0,2.9
1,belem,2025-01-01 01:00:00,23.9,98.0,0.0,1.6
2,belem,2025-01-01 02:00:00,23.9,99.0,0.0,3.3
3,belem,2025-01-01 03:00:00,23.8,98.0,0.0,3.5
4,belem,2025-01-01 04:00:00,23.9,97.0,0.1,4.4
